# Local / Spot Optimization — Targeting Specific Prompt Sections

By default APO rewrites your **entire** prompt. With **local (spot) optimization** you tell APO to optimize only the part you choose, and to leave the rest exactly as-is, using two reserved tags in the `advpo:` namespace:

- `<advpo:optimize>...</advpo:optimize>` — the **focus area**. The optimizer rewrites only what is inside these tags.
- `<advpo:exclude>...</advpo:exclude>` — a **frozen** region. The optimizer will not touch it.

Rules of thumb:
- If **any** `<advpo:optimize>` tag is present, only the tagged regions are rewritten; everything else is preserved automatically (so you don't also need `<advpo:exclude>`).
- If you use **only** `<advpo:exclude>`, everything outside the excluded regions is optimized.
- The returned prompt is **clean** — the tags are stripped from the optimized output.

This notebook reuses **steering** mode (no Lambda, no judge) so the focus stays on the tags. Set `APO_USE_REFERENCE=1` to replay from bundled results.

In [ ]:
import os
from pathlib import Path
from IPython.display import display, Markdown

import utils

env = utils.load_env()
bedrock, s3, lambda_client = utils.make_clients(env)

# Default target model matches the bundled reference run. Edit and re-execute to try other models.
TARGET_MODEL_ID = "us.anthropic.claude-opus-4-6-v1"

USING_REPLAY = os.environ.get("APO_USE_REFERENCE") == "1"
display(Markdown(f"**Live mode:** `{not USING_REPLAY}`  |  **Bucket:** `{env['BUCKET']}`  |  **Region:** `{env['REGION']}`"))

## The tagged prompt template

The template below freezes the policy line with `<advpo:exclude>` and marks the (deliberately weak) response instruction with `<advpo:optimize>` as the focus area. `{{message}}` is a normal input variable, substituted at runtime; APO preserves it.

In [ ]:
template_text = (Path('data/spot/prompt_template.txt')).read_text()
print(template_text)

## Build the record

Steering criteria describe what a good reply looks like; the optimizer enforces them while rewriting **only** the `<advpo:optimize>` region.

In [ ]:
SPOT_CRITERIA = [
    "Acknowledge the customer's specific concern in the first sentence.",
    "Provide exactly one concrete next step or resolution.",
    "Keep the reply under 80 words.",
    "Maintain a warm, professional tone with no internal jargon.",
]

record = utils.build_steering_record("spot", steering_criteria=SPOT_CRITERIA, bucket=env["BUCKET"])
display(utils.render_record_shape(record))
display(Markdown("**First sample shape:**"))
display(utils.render_sample_shape(record))

> **Heads up — long-running cell.** In live mode this submits a real APO job that can take ~20 minutes. The `status` / `elapsed` line refreshes until the job reaches a terminal state. With `APO_USE_REFERENCE=1` it returns instantly from the bundled results.

In [ ]:
handle = display(Markdown("_waiting for status…_"), display_id=True)
def on_status(status, elapsed_s):
    handle.update(Markdown(f"**status:** `{status}`  |  **elapsed:** `{elapsed_s:.0f}s`"))

spot_results = utils.run_job(
    record,
    example="spot",
    model_id=TARGET_MODEL_ID,
    env=env,
    on_status=on_status,
)
print(f"results file: {spot_results}")

In [ ]:
parsed = utils.parse_results(spot_results)
display(utils.render_results_table(parsed))
for row in parsed:
    display(utils.render_prompt_diff(row))

## What to look for in the diff

Compare the **original** vs **optimized** templates above and confirm:

1. The `<advpo:exclude>` policy line is **unchanged** (frozen).
2. Only the text that was inside `<advpo:optimize>` is **rewritten** — the weak "write a helpful reply" instruction becomes a more structured one shaped by the steering criteria.
3. The `{{message}}` variable is **preserved**.
4. The optimized prompt is **clean** — the `<advpo:optimize>` / `<advpo:exclude>` tags are **stripped** from the output.

That's local/spot optimization: improve one section, leave the rest exactly as you wrote it.